In [ ]:
%config IPCompleter.use_jedi = False
%pdb off
%load_ext autoreload
%autoreload 3

%matplotlib qt
# %matplotlib notebook
import matplotlib.pyplot as plt

# %matplotlib notebook
from pathlib import Path
import numpy as np
import shutil
import sys
from typing import List, Dict, Tuple, Union, Set, Optional

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import sys
import os
from pathlib import Path

from neuropy.io.neuroscopeio import NeuroscopeIO
from neuropy.io.binarysignalio import BinarysignalIO 
from neuropy.io.miniscopeio import MiniscopeIO

# from neuropy.core import Epoch, ProcessData
from neuropy.core.epoch import Epoch, EpochHelpers, ensure_dataframe, ensure_Epoch
from neuropy.io.spykingcircusio import SpykingCircusIO
from neuropy.core.session.data_session_loader import DataSessionLoader
from neuropy.core.session.init_from_raw_data import RawDataInitializationMixin, windows_to_wsl_path_if_needed, find_first_file_rglob

from typing import List, Dict, Tuple, Union, Set
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.full as si
from spikeinterface.core import write_binary_recording

def concatenate_to_output_file(input_paths: List[Path], output_path: Path):
    """ concatenates the binary files in input_paths to a single joined output_path. 
    """
    with open(output_path, "wb") as outfile:
        for path in input_paths:
            with open(path, "rb") as infile:
                shutil.copyfileobj(infile, outfile)
            

def step_perform_concat(found_raw_data_paths: List[Path], spyk_circ_output_dir: Path):
    """ performs the concatenation, creates the output directory if needed
    """
    ## make the "spyk-circ" output directory
    # spyk_circ_output_dir: Path = Path('W:/Data/Bapun/RatS/Day1Openfield/spyk-circ').resolve()
    spyk_circ_output_dir.mkdir(exist_ok=True, parents=True) ## dang I sure hope we're on Windows or I'll add some garbage paths :P
    
    ## Copy the concatenated files to the output directory
    concatenated_file_output_path: Path = spyk_circ_output_dir.joinpath('continuous_combined.dat').resolve() ## do I need to do anything with the adjacent `timestamps.npy` or anything??
    if not concatenated_file_output_path.exists():
        concatenate_to_output_file(input_paths=found_raw_data_paths, output_path=concatenated_file_output_path)
    return concatenated_file_output_path


def step_perform_downsample(concatenated_file_output_path: Path, sampling_frequency=30000, num_chan=200, resample_rate=1250) -> Path:
    # 1. Map the raw binary file (doesn't load into RAM yet)
    recording = se.read_binary(
        file_paths=concatenated_file_output_path.as_posix(), 
        sampling_frequency=sampling_frequency, 
        n_channels=num_chan, 
        dtype='int16'
    )

    # 2. Set up the downsampling node (applies anti-aliasing filter automatically)
    recording_lfp = spre.resample(recording, resample_rate=resample_rate)

    output_lfp_path: Path = concatenated_file_output_path.with_suffix('.lfp').resolve()
    print(f'trying to write to: "{output_lfp_path.as_posix()}"')
    # 3. Execute and write to a flat binary file in chunks
    # n_jobs=-1 uses all CPU cores for faster processing
    write_binary_recording(
        recording_lfp, 
        file_paths=output_lfp_path.as_posix(), 
        dtype='int16', 
        n_jobs=-1, 
        chunk_duration='1s'
    )
    print(f'\tdone.')
    return output_lfp_path


In [ ]:
import spikeinterface.full as si


## Bapun Format:
# basedir = Path('/home/halechr/FastData/Bapun/RatS/Day1Openfield') # Linux
basedir = windows_to_wsl_path_if_needed(Path('W:/Data/Bapun/RatS/Day1Openfield')) # Windows
# basedir = Path("/mnt/w/Data/Bapun/RatS/Day1Openfield") # Apogee WSL
# basedir = Path('/Volumes/iNeo/Data/Bapun/Day5TwoNovel') # MacOS

# n_channels: int = 200
n_channels: int = 195
dat_file_sampling_rate: int = 30000
eeg_sampling_rate: int = 1250
basename: str = 'RatS-Day1Openfield'
excluded_data_datetimes = ['2020-11-25_10-24-24', '2020-11-25_15-06-02']

xml_path: Path = find_first_file_rglob(basedir, '*.xml', recursive=False)
xml_path
print(f'xml_path: {xml_path}')


In [ ]:
# recording = si.read_XXXX('/path/to/my/recording')
# full_dat_file = windows_to_wsl_path_if_needed(Path(r"W:/Data/Bapun/RatS/Day1Openfield/RatS-Day1Openfield.dat").resolve())
full_dat_file: Path = windows_to_wsl_path_if_needed(find_first_file_rglob(basedir, '*.dat', recursive=False))
print(f'full_dat_file: {full_dat_file.as_posix()}')
assert full_dat_file.exists(), f"full_dat_file: {full_dat_file} does not exist."


In [ ]:
spiking_circus_dir: Path = windows_to_wsl_path_if_needed(basedir.joinpath('spyk-circ', basename).resolve())
assert spiking_circus_dir.exists(), f"spiking_circus_dir: {spiking_circus_dir} does not exist!"
# 1. Map the raw binary file (doesn't load into RAM yet)
# se.read_phy
spiking_circus_loaded = se.read_spykingcircus(spiking_circus_dir)

spiking_circus_loaded

In [ ]:
n_channels

In [ ]:
recording = se.read_binary(
    file_paths=full_dat_file.as_posix(), 
    sampling_frequency=dat_file_sampling_rate, 
    num_chan=n_channels, 
    dtype='int16'
)
recording

In [ ]:

recording_filtered = si.bandpass_filter(recording)
sorting = si.run_sorter('YYYYY', recording_filtered)


job_kwargs = dict(n_jobs=-1, progress_bar=True, chunk_duration="1s")

# make the SortingAnalyzer with necessary and some optional extensions
sorting_analyzer = si.create_sorting_analyzer(sorting, recording_filtered,
                                              format="binary_folder", folder="/my_sorting_analyzer",
                                              **job_kwargs)
sorting_analyzer.compute("random_spikes", method="uniform", max_spikes_per_unit=500)
sorting_analyzer.compute("waveforms", **job_kwargs)
sorting_analyzer.compute("templates", **job_kwargs)
sorting_analyzer.compute("noise_levels")
sorting_analyzer.compute("unit_locations", method="monopolar_triangulation")
sorting_analyzer.compute("isi_histograms")
sorting_analyzer.compute("correlograms", window_ms=100, bin_ms=5.)
sorting_analyzer.compute("principal_components", n_components=3, mode='by_channel_global', whiten=True, **job_kwargs)
sorting_analyzer.compute("quality_metrics", metric_names=["snr", "firing_rate"])
sorting_analyzer.compute("template_similarity")
sorting_analyzer.compute("spike_amplitudes", **job_kwargs)

In [ ]:
print("Hello from rats!")
raw_data_path: Path = Path(r'W:/Data/Bapun/RatS/Day1Openfield/Raw_data').resolve()
print(f'raw_data_path: "{raw_data_path.as_posix()}"')

## find all constitutent "continuous.dat" files recurrsively in all subdirectories: "W:\Data\Bapun\RatS\Day1Openfield\Raw_data\2020-11-25_10-24-24\experiment1\recording1\continuous\Rhythm_FPGA-100.0\continuous.dat"
found_raw_data_paths = ["W:/Data/Bapun/RatS/Day1Openfield/Raw_data/2020-11-25_10-20-27/experiment1/recording1/continuous/Rhythm_FPGA-100.0/continuous.dat",
                        # "W:/Data/Bapun/RatS/Day1Openfield/Raw_data/2020-11-25_10-24-24/experiment1/recording1/continuous/Rhythm_FPGA-100.0/continuous.dat", ## BAD ONE, only has 32 channels, skip
                        "W:/Data/Bapun/RatS/Day1Openfield/Raw_data/2020-11-25_13-02-47/experiment1/recording1/continuous/Rhythm_FPGA-100.0/continuous.dat",
                        "W:/Data/Bapun/RatS/Day1Openfield/Raw_data/2020-11-25_14-30-32/experiment1/recording1/continuous/Rhythm_FPGA-100.0/continuous.dat",
                        "W:/Data/Bapun/RatS/Day1Openfield/Raw_data/2020-11-25_15-06-02/experiment1/recording1/continuous/Rhythm_FPGA-100.0/continuous.dat",
]    
## *-24-24 is a bad one with only 30 good channels!

## Iterate through and make proper paths, check their existance
found_raw_data_paths: List[Path] = [Path(v).resolve() for v in found_raw_data_paths]
## could assert that they all exist... but let's NOT!

## make the "spyk-circ" output directory
spyk_circ_output_dir: Path = Path('W:/Data/Bapun/RatS/Day1Openfield/spyk-circ').resolve()
spyk_circ_output_dir.mkdir(exist_ok=True, parents=True) ## dang I sure hope we're on Windows or I'll add some garbage paths :P

## Copy the concatenated files to the output directory
concatenated_file_output_path: Path = step_perform_concat(found_raw_data_paths=found_raw_data_paths, spyk_circ_output_dir=spyk_circ_output_dir)
print(f'have concatenated_file_output_path: "{concatenated_file_output_path.as_posix()}"')
concatenated_file_output_path


In [ ]:

output_lfp_path = step_perform_downsample(concatenated_file_output_path=concatenated_file_output_path)

## try specific output path
# output_lfp_path = step_perform_downsample(concatenated_file_output_path=Path(r'W:\Data\Bapun\RatS\Day1Openfield\Raw_data\2020-11-25_13-02-47\experiment1\recording1\continuous\Rhythm_FPGA-100.0\continuous.dat'), num_chan=192)
print(f'done with saving downsampled lfp')

In [ ]:
dat_file_sampling_rate: int = 30000 
num_chan: int = 200
eeg_sampling_rate: int = 1250

# 1. Map the raw binary file (doesn't load into RAM yet)
recording = se.read_binary(
    file_paths=[concatenated_file_output_path.as_posix()], 
    sampling_frequency=dat_file_sampling_rate, 
    # channel_ids=np.arange(num_chan),
    # num_chan=num_chan, 
    dtype='int16'
)
recording

In [ ]:

# 2. Set up the downsampling node (applies anti-aliasing filter automatically)
recording_lfp = spre.resample(recording, resample_rate=eeg_sampling_rate)

output_lfp_path: Path = concatenated_file_output_path.with_suffix('.lfp').resolve()
print(f'trying to write to: "{output_lfp_path.as_posix()}"')
# 3. Execute and write to a flat binary file in chunks
# n_jobs=-1 uses all CPU cores for faster processing
write_binary_recording(
    recording_lfp, 
    file_paths=output_lfp_path.as_posix(), 
    dtype='int16', 
    n_jobs=-1, 
    chunk_duration='1s'
)
print(f'\tdone.')